# Module 6: MCP — From Consumer to Producer

**Day 4 — Agents, LangGraph & MCP**

## What you will learn
- What MCP is and how JSON-RPC 2.0 works
- **Consumer** (Week 3): connecting to MCP servers
- **Producer** (this module): building your own MCP server
- Wrapping a Spring Boot REST API as an MCP tool
- Context window management with `tiktoken`

## 1. What is MCP?

MCP (Model Context Protocol) is an open standard for LLMs to call external tools. It uses **JSON-RPC 2.0** as the wire protocol.

```
LLM Agent  ←── JSON-RPC 2.0 ──→  MCP Server  ←── HTTP/DB/files ──→  Your Services
```

Think of it like REST for AI tools — any service that speaks MCP can plug into any MCP-compatible LLM.

## 2. JSON-RPC 2.0 — The Wire Format

Every MCP call is a JSON-RPC 2.0 message. Let's trace a full request/response.

In [ ]:
import json

# 1. Client (LLM agent) sends a tool call:
rpc_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {"name": "get_weather", "arguments": {"city": "Mumbai"}}
}

# 2. Server executes the tool:
def get_weather(city: str) -> dict:
    return {"city": city, "temperature": 32, "unit": "celsius", "description": "Hot and humid"}

tool_result = get_weather(**rpc_request["params"]["arguments"])

# 3. Server returns the result:
rpc_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {"content": [{"type": "text", "text": json.dumps(tool_result)}]}
}

print("Request:")
print(json.dumps(rpc_request, indent=2))
print("\nResponse:")
print(json.dumps(rpc_response, indent=2))

## 3. Tool Discovery — tools/list

Before calling any tool, the LLM asks: *"what tools do you have?"*

In [ ]:
tool_schema = {
    "name": "get_weather",
    "description": "Get current weather for a city. Use when the user asks about weather or temperature.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. Mumbai, Delhi, London"}
        },
        "required": ["city"]
    }
}
print("Tool schema exposed by MCP server:")
print(json.dumps(tool_schema, indent=2))
print("\n→ The LLM reads this to decide WHEN and HOW to call the tool.")

## 4. Building an MCP Server (Producer)

Week 3: you *consumed* MCP servers (connected as client).
Now: you *build* one — you become the server.

```python
# pip install mcp
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-service")

@mcp.tool()          # Callable action — side effects OK
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return f"Weather in {city}: 22°C, Sunny"

@mcp.resource("data://items/{id}")  # Read-only data
def get_item(id: str) -> str:
    return f"Item {id} data"

@mcp.prompt()        # Reusable prompt template
def analysis_prompt(city: str) -> str:
    return f"Analyse the weather in {city} and suggest clothing."

if __name__ == "__main__":
    mcp.run()  # stdio transport (default for CLI tools)
    # For web: mcp.run(transport='sse', host='0.0.0.0', port=8000)
```

## 5. Wrapping a Spring Boot Service as MCP

Your existing Spring Boot REST APIs become MCP tools — no changes to Spring Boot needed.

In [ ]:
spring_wrapper = '''
from mcp.server.fastmcp import FastMCP
import httpx

mcp = FastMCP("spring-boot-wrapper")
SPRING_URL = "http://localhost:8080"

@mcp.tool()
async def get_products(category: str) -> list:
    """Get products from the inventory service (Spring Boot backend)."""
    async with httpx.AsyncClient() as client:
        r = await client.get(f"{SPRING_URL}/api/products",
                             params={"category": category})
        return r.json()

@mcp.tool()
async def create_order(product_id: str, quantity: int) -> dict:
    """Create an order via the order service (Spring Boot backend)."""
    async with httpx.AsyncClient() as client:
        r = await client.post(f"{SPRING_URL}/api/orders",
                              json={"productId": product_id, "quantity": quantity})
        return r.json()

if __name__ == "__main__":
    mcp.run()  # Now your Spring Boot API speaks MCP!
'''
print(spring_wrapper)

## 6. Context Window Management with tiktoken

Before sending documents to an LLM, count tokens to avoid exceeding the context limit.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def fit_to_context(docs: list[str], max_tokens: int = 3000) -> list[str]:
    """Keep as many docs as fit within the token budget."""
    selected, used = [], 0
    for doc in docs:
        t = count_tokens(doc)
        if used + t > max_tokens:
            break
        selected.append(doc)
        used += t
    print(f"Selected {len(selected)}/{len(docs)} docs using {used} tokens (limit: {max_tokens})")
    return selected

docs = ["This is document " + str(i) + ". " + "content " * 50 for i in range(20)]
fitted = fit_to_context(docs, max_tokens=500)

## 7. Using day4 modules

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
from day4.mcp_integration import (
    show_mcp_producer_code, show_json_rpc_example,
    show_spring_boot_mcp_wrapper, tokenize_and_count
)

for text in [
    "Hello world",
    "Explain how transformers use attention mechanisms",
    "x" * 500,
]:
    r = tokenize_and_count(text)
    print(f"  {r['token_count']:>6} tokens | {r['text_preview']}")

In [ ]:
from day4.mcp_integration import mcp_filesystem_read, mcp_github_create_issue, show_mcp_servers

print("Mock MCP tools:")
print("  filesystem:", mcp_filesystem_read.invoke({"file_path": "README.md"})[:60])
print("  github:    ", mcp_github_create_issue.invoke({"title": "Test", "body": "test", "repo": "my/repo"}))

print("\nAvailable MCP server catalogue:")
for s in show_mcp_servers():
    print(f"  {s['name']:<20} {s.get('description', '')[:50]}")

In [ ]:
from day4.mcp_integration import build_mcp_agent
from day4.tools_agents import MockLLMWithTools
from langchain_core.messages import HumanMessage

llm   = MockLLMWithTools("mcp_filesystem_read", {"file_path": "config.yaml"}, "File read successfully")
agent = build_mcp_agent(llm, mock_mode=True)
result = agent.invoke({"messages": [HumanMessage("Read the config file")]})
print("Agent output:", result["messages"][-1].content)